In [31]:
import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE



In [32]:
df = pd.read_csv(r'C:\Employee_attrition_classification\data\test.csv')

df.shape

(14900, 24)

In [33]:
df.isnull().sum()

Employee ID                 0
Age                         0
Gender                      0
Years at Company            0
Job Role                    0
Monthly Income              0
Work-Life Balance           0
Job Satisfaction            0
Performance Rating          0
Number of Promotions        0
Overtime                    0
Distance from Home          0
Education Level             0
Marital Status              0
Number of Dependents        0
Job Level                   0
Company Size                0
Company Tenure              0
Remote Work                 0
Leadership Opportunities    0
Innovation Opportunities    0
Company Reputation          0
Employee Recognition        0
Attrition                   0
dtype: int64

In [34]:
df.drop(columns = 'Employee ID',inplace = True)


In [35]:
from collections import OrderedDict

stats = []

for i in df.select_dtypes(exclude = "object"):
    numerical_stats = OrderedDict({
    'Feature' : i,
    'Count' : df[i].count(),
    'Mean' : df[i].mean(),
    'Median' : df[i].median(),
    'Maximum' : df[i].max(),
    'Minimum' : df[i].min(),
    'Q1' : df[i].quantile(0.25),
    'Q3' : df[i].quantile(0.75),
    'IQR' : df[i].quantile(0.75),
    'Missing value%' : (df[i].isnull().sum() / len(df)) * 100,
    'Standard Deviation' : df[i].std(),
    'Skewness' : df[i].skew(),
    'Kurtosis' : df[i].kurtosis()
    })
    stats.append(numerical_stats)
    report = pd.DataFrame(stats)

report

,Feature,Count,Mean,Median,Maximum,Minimum,Q1,Q3,IQR,Missing value%,Standard Deviation,Skewness,Kurtosis
0,Age,14900,38.385235,38.0,59,18,28.00,49.0,49.0,0.0,12.097904,0.020160,-1.190540
1,Years at Company,14900,15.592416,13.0,51,1,7.00,23.0,23.0,0.0,11.133792,0.796660,-0.084650
2,Monthly Income,14900,7287.306040,7332.0,15063,1226,5633.75,8852.0,8852.0,0.0,2156.737934,0.131206,-0.588337
3,Number of Promotions,14900,0.834362,1.0,4,0,0.00,2.0,2.0,0.0,0.996511,0.990370,0.169052
4,Distance from Home,14900,49.927315,50.0,99,1,25.00,75.0,75.0,0.0,28.702307,-0.002473,-1.206855
5,Number of Dependents,14900,1.659329,1.0,6,0,0.00,3.0,3.0,0.0,1.545401,0.673002,-0.542063
6,Company Tenure,14900,55.603624,56.0,127,2,36.00,75.0,75.0,0.0,25.352807,0.061507,-0.776650


In [36]:
X = df.drop(columns = ['Attrition'])
y = df['Attrition']

numerical_cols = X.select_dtypes(exclude="object").columns
categorical_cols = X.select_dtypes(include="object").columns

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=1)

numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler())
    ])

categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False
        ))
    ])

transformer = ColumnTransformer([
        ("num", numerical_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

X_train = transformer.fit_transform(X_train)
X_test = transformer.transform(X_test)

sm=SMOTE()
X_train,y_train=sm.fit_resample(X_train,y_train)

C:\Users\sanke\AppData\Local\Temp\ipykernel_3204\1060998390.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include="object").columns


In [37]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=1)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)


In [38]:
df.shape

(14900, 23)